# Loan Default Prediction - Complete Pipeline

This notebook demonstrates the complete workflow from data loading to model evaluation.

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_curve, roc_auc_score
)
import warnings
warnings.filterwarnings('ignore')

print("✓ All libraries imported successfully!")

## 2. Load and Explore Data

In [ ]:
# Load dataset
df = pd.read_csv('../data/dataset.csv')

print("="*60)
print("DATASET OVERVIEW")
print("="*60)
print(f"Dataset shape: {df.shape}")
print(f"\nFirst few rows:")
display(df.head())

In [ ]:
print("\n" + "="*60)
print("DATA INFO")
print("="*60)
print(f"\nData types:\n{df.dtypes}")
print(f"\nMissing values:\n{df.isnull().sum()}")
print(f"\nBasic statistics:")
display(df.describe())

In [ ]:
print("\n" + "="*60)
print("TARGET DISTRIBUTION")
print("="*60)
print(f"\nDefault distribution:")
print(df['Default'].value_counts())
print(f"\nClass balance:")
print(df['Default'].value_counts(normalize=True) * 100)

## 3. Data Preprocessing

In [ ]:
print("="*60)
print("DATA PREPROCESSING")
print("="*60)

# Separate features and target
target_col = "Default"
exclude_cols = ["LoanID", target_col]

X = df.drop(columns=exclude_cols)
y = df[target_col]

print(f"\nFeatures shape: {X.shape}")
print(f"Target shape: {y.shape}")

# One-hot encode categorical variables if any
X_encoded = pd.get_dummies(X, drop_first=True)
print(f"\nAfter encoding: {X_encoded.shape}")

In [ ]:
# Train-test split with stratification
X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train set: {X_train.shape}")
print(f"Test set: {X_test.shape}")
print(f"\nTrain target distribution:")
print(y_train.value_counts())

In [ ]:
# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"\n✓ Features scaled successfully!")
print(f"\nScaled train set shape: {X_train_scaled.shape}")
print(f"Scaled test set shape: {X_test_scaled.shape}")

## 4. Model Training

### 4.1 Naive Baseline Model

In [ ]:
print("="*60)
print("NAIVE BASELINE MODEL")
print("="*60)

# Naive baseline: always predict majority class
majority_class = y_train.mode()[0]
y_pred_naive = np.array([majority_class] * len(y_test))

print(f"\nNaive strategy: Always predict class {majority_class}")
print(f"Majority class percentage in training: {(y_train == majority_class).mean() * 100:.2f}%")

### 4.2 Logistic Regression Model

In [ ]:
print("="*60)
print("LOGISTIC REGRESSION MODEL")
print("="*60)

# Train Logistic Regression with balanced class weights
model = LogisticRegression(
    max_iter=2000,
    class_weight='balanced',  # Handle class imbalance
    random_state=42
)

model.fit(X_train_scaled, y_train)

print("\n✓ Logistic Regression model trained successfully!")
print(f"Number of features: {len(model.coef_[0])}")

In [ ]:
# Get predictions and probabilities
y_pred_lr = model.predict(X_test_scaled)
y_prob_lr = model.predict_proba(X_test_scaled)[:, 1]

# For naive baseline
y_prob_naive = np.array([0.0 if p == 0 else 1.0 for p in y_pred_naive])

print("\n✓ Predictions generated!")

## 5. Model Evaluation

### 5.1 Naive Baseline Metrics

In [ ]:
print("="*60)
print("NAIVE BASELINE - PERFORMANCE METRICS")
print("="*60)

accuracy_naive = accuracy_score(y_test, y_pred_naive)
precision_naive = precision_score(y_test, y_pred_naive, zero_division=0)
recall_naive = recall_score(y_test, y_pred_naive, zero_division=0)
f1_naive = f1_score(y_test, y_pred_naive, zero_division=0)

print(f"\nAccuracy:  {accuracy_naive:.4f}")
print(f"Precision: {precision_naive:.4f}")
print(f"Recall:    {recall_naive:.4f}")
print(f"F1 Score:  {f1_naive:.4f}")

print("\nConfusion Matrix:")
cm_naive = confusion_matrix(y_test, y_pred_naive)
print(cm_naive)
print(f"\nTN: {cm_naive[0,0]}, FP: {cm_naive[0,1]}")
print(f"FN: {cm_naive[1,0]}, TP: {cm_naive[1,1]}")

In [ ]:
print("\nClassification Report - Naive Baseline:")
print(classification_report(y_test, y_pred_naive, zero_division=0))

### 5.2 Logistic Regression Metrics

In [ ]:
print("="*60)
print("LOGISTIC REGRESSION - PERFORMANCE METRICS")
print("="*60)

accuracy_lr = accuracy_score(y_test, y_pred_lr)
precision_lr = precision_score(y_test, y_pred_lr, zero_division=0)
recall_lr = recall_score(y_test, y_pred_lr, zero_division=0)
f1_lr = f1_score(y_test, y_pred_lr, zero_division=0)
roc_auc_lr = roc_auc_score(y_test, y_prob_lr)

print(f"\nAccuracy:  {accuracy_lr:.4f}")
print(f"Precision: {precision_lr:.4f}")
print(f"Recall:    {recall_lr:.4f}")
print(f"F1 Score:  {f1_lr:.4f}")
print(f"ROC-AUC:   {roc_auc_lr:.4f}")

print("\nConfusion Matrix:")
cm_lr = confusion_matrix(y_test, y_pred_lr)
print(cm_lr)
print(f"\nTN: {cm_lr[0,0]}, FP: {cm_lr[0,1]}")
print(f"FN: {cm_lr[1,0]}, TP: {cm_lr[1,1]}")

In [ ]:
print("\nClassification Report - Logistic Regression:")
print(classification_report(y_test, y_pred_lr, zero_division=0))

### 5.3 Model Comparison

In [ ]:
print("="*60)
print("MODEL COMPARISON")
print("="*60)

comparison_df = pd.DataFrame({
    'Metric': ['Accuracy', 'Precision', 'Recall', 'F1 Score', 'ROC-AUC'],
    'Naive Baseline': [accuracy_naive, precision_naive, recall_naive, f1_naive, 0.0],
    'Logistic Regression': [accuracy_lr, precision_lr, recall_lr, f1_lr, roc_auc_lr]
})

comparison_df['Improvement'] = comparison_df['Logistic Regression'] - comparison_df['Naive Baseline']
comparison_df['Improvement %'] = (comparison_df['Improvement'] / (comparison_df['Naive Baseline'].abs() + 1e-6)) * 100

print("\n" + comparison_df.to_string(index=False))

## 6. Visualizations

### 6.1 Confusion Matrices Comparison

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Naive Baseline CM
sns.heatmap(cm_naive, annot=True, fmt='d', cmap='Blues', ax=axes[0], cbar=False)
axes[0].set_title('Naive Baseline - Confusion Matrix', fontsize=12, fontweight='bold')
axes[0].set_ylabel('True Label')
axes[0].set_xlabel('Predicted Label')

# Logistic Regression CM
sns.heatmap(cm_lr, annot=True, fmt='d', cmap='Greens', ax=axes[1], cbar=False)
axes[1].set_title('Logistic Regression - Confusion Matrix', fontsize=12, fontweight='bold')
axes[1].set_ylabel('True Label')
axes[1].set_xlabel('Predicted Label')

plt.tight_layout()
plt.show()

### 6.2 Metrics Comparison Bar Plot

In [ ]:
metrics = ['Accuracy', 'Precision', 'Recall', 'F1 Score']
naive_scores = [accuracy_naive, precision_naive, recall_naive, f1_naive]
lr_scores = [accuracy_lr, precision_lr, recall_lr, f1_lr]

x = np.arange(len(metrics))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 6))
bars1 = ax.bar(x - width/2, naive_scores, width, label='Naive Baseline', alpha=0.8)
bars2 = ax.bar(x + width/2, lr_scores, width, label='Logistic Regression', alpha=0.8)

ax.set_xlabel('Metrics', fontsize=12, fontweight='bold')
ax.set_ylabel('Score', fontsize=12, fontweight='bold')
ax.set_title('Model Comparison - Performance Metrics', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(metrics)
ax.legend()
ax.set_ylim([0, 1])
ax.grid(axis='y', alpha=0.3)

# Add value labels on bars
for bars in [bars1, bars2]:
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
                f'{height:.3f}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

### 6.3 ROC Curve - Logistic Regression

In [ ]:
fpr, tpr, thresholds = roc_curve(y_test, y_prob_lr)

fig, ax = plt.subplots(figsize=(8, 6))
ax.plot(fpr, tpr, label=f'Logistic Regression (AUC = {roc_auc_lr:.4f})', linewidth=2, color='blue')
ax.plot([0, 1], [0, 1], linestyle='--', color='gray', linewidth=2, label='Random Classifier')

ax.set_xlabel('False Positive Rate', fontsize=12, fontweight='bold')
ax.set_ylabel('True Positive Rate', fontsize=12, fontweight='bold')
ax.set_title('ROC Curve - Logistic Regression', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(alpha=0.3)
ax.set_xlim([0, 1])
ax.set_ylim([0, 1])

plt.tight_layout()
plt.show()

## 7. Key Insights & Recommendations

In [ ]:
print("="*60)
print("SUMMARY & RECOMMENDATIONS")
print("="*60)

print("\n📊 KEY FINDINGS:")
print(f"\n1. Naive Baseline Performance:")
print(f"   - Achieves {accuracy_naive:.2%} accuracy by always predicting majority class")
print(f"   - Cannot identify any defaults (F1 Score: {f1_naive:.4f})")
print(f"   - Establishes a minimum performance threshold")

print(f"\n2. Logistic Regression Performance:")
print(f"   - F1 Score: {f1_lr:.4f} (vs Naive: {f1_naive:.4f})")
print(f"   - Successfully identifies defaults with {recall_lr:.2%} recall")
print(f"   - ROC-AUC: {roc_auc_lr:.4f} - Good discriminative ability")

print(f"\n3. Class Imbalance Impact:")
print(f"   - Using class_weight='balanced' in LogisticRegression")
print(f"   - Handles imbalanced data effectively")
print(f"   - Improved recall by sacrificing some accuracy (expected trade-off)")

print(f"\n🎯 RECOMMENDATIONS FOR IMPROVEMENT:")
print(f"\n1. Try Advanced Models:")
print(f"   - Random Forest (non-linear relationships)")
print(f"   - XGBoost or LightGBM (better performance on imbalanced data)")
print(f"   - Neural Networks (complex patterns)")

print(f"\n2. Feature Engineering:")
print(f"   - Create interaction terms")
print(f"   - Polynomial features")
print(f"   - Domain-specific features based on business logic")

print(f"\n3. Handle Class Imbalance:")
print(f"   - Apply SMOTE (Synthetic Minority Oversampling)")
print(f"   - Try different class weight ratios")
print(f"   - Ensemble methods")

print(f"\n4. Hyperparameter Tuning:")
print(f"   - Use GridSearchCV or RandomizedSearchCV")
print(f"   - Cross-validation for stable estimates")
print(f"   - Different decision thresholds")

print(f"\n5. Business Considerations:")
print(f"   - False Positives cost: Unnecessary loan rejections")
print(f"   - False Negatives cost: Actual defaults not caught")
print(f"   - Adjust threshold based on business priorities")

print("\n" + "="*60)